In [11]:
# !conda install gspread-dataframe gspread

In [ ]:
import gspread

try:
    gc = gspread.service_account(filename="credentials.json")
    print("✅ Авторизация прошла успешно!")

    print("Список доступных таблиц:")
    for sheet in gc.openall():
        print("-", sheet.title)

except Exception as e:
    print("❌ Ошибка при авторизации:")
    print(e)

sheet_name = "abc"
try:
    sh = gc.open(sheet_name)
except Exception as e:
    print(f"❌ Таблица с именем {sheet_name} не найдена")

✅ Авторизация прошла успешно!
Список доступных таблиц:
- Баскетбольный клуб смета
<Response [200]>


In [10]:
import gspread
from gspread_dataframe import get_as_dataframe
import pandas as pd
import time

# Авторизация
gc = gspread.service_account(filename='credentials.json')

# Открываем таблицу по имени
sh = gc.open("Баскетбольный клуб смета")
worksheet = sh.sheet1  # или sh.worksheet("Лист1")

# Храним номер последней строки, которую обработали
last_row = 0

def check_for_new_rows():
    global last_row
    # Загружаем все данные как DataFrame
    df = get_as_dataframe(worksheet, evaluate_formulas=True).dropna(how="all")
    current_row_count = len(df)

    if current_row_count > last_row:
        # Есть новые строки
        new_rows = df.iloc[last_row:current_row_count]
        for index, row in new_rows.iterrows():
            print(f"Новая строка: {row.to_dict()}")

        # Обновляем индекс последней строки
        last_row = current_row_count
    else:
        print("Новых строк нет.")

# Пример: отслеживать изменения каждые 10 секунд
if __name__ == "__main__":
    while True:
        check_for_new_rows()
        time.sleep(10)


Новая строка: {'Unnamed: 1': 'Название позиции', 'Unnamed: 2': 'Ссылка', 'Unnamed: 3': 'Стоимость единицы', 'Unnamed: 4': 'Количество ', 'Unnamed: 5': 'Сумма', 'Unnamed: 6': nan}
Новая строка: {'Unnamed: 1': 'Вступительный взнос в лигу', 'Unnamed: 2': 'https://vk.com/newbasketleague', 'Unnamed: 3': 6000, 'Unnamed: 4': 1, 'Unnamed: 5': 6000, 'Unnamed: 6': nan}
Новая строка: {'Unnamed: 1': 'Взнос за игры', 'Unnamed: 2': nan, 'Unnamed: 3': 6800, 'Unnamed: 4': 16, 'Unnamed: 5': 126600, 'Unnamed: 6': 'Игры каждую неделю по выходным. В одном сезоне от 16 игр, полная стоимость с учетом налога 132600'}
Новая строка: {'Unnamed: 1': 'Тактическая доска', 'Unnamed: 2': 'https://www.ozon.ru/product/takticheskaya-doska-dlya-basketbola-1343143927/?at=r2t4QnP9PhxWGEMXfQJxqXzSZ94l0xtVwAXy1tkyXkkk&keywords=%D1%82%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F+%D0%B4%D0%BE%D1%81%D0%BA%D0%B0+%D0%B4%D0%BB%D1%8F+%D0%B1%D0%B0%D1%81%D0%BA%D0%B5%D1%82%D0%B1%D0%BE%D0%BB%D0%B0', 'Unnamed: 3': 800, '

KeyboardInterrupt: 

In [ ]:
import asyncio
from aiogram import Bot, Dispatcher, types
import gspread
from gspread_dataframe import get_as_dataframe

TOKEN = 'your-telegram-bot-token'
bot = Bot(token=TOKEN)
dp = Dispatcher()

# Подключение к Google Sheets
gc = gspread.service_account(filename='credentials.json')
worksheet = gc.open("Имя вашей таблицы").sheet1

last_row = 0

# Асинхронный парсер
async def sheet_watcher():
    global last_row
    while True:
        df = get_as_dataframe(worksheet).dropna(how='all')
        if len(df) > last_row:
            new_rows = df.iloc[last_row:]
            for _, row in new_rows.iterrows():
                message = f"Новая строка: {row.to_dict()}"
                print(message)
                await bot.send_message(chat_id='ВАШ_CHAT_ID', text=message)
            last_row = len(df)
        await asyncio.sleep(10)  # интервал проверки

# Хендлер команды
@dp.message(commands=['start'])
async def cmd_start(message: types.Message):
    await message.answer("Привет! Я работаю и отслеживаю Google Таблицу.")

# Запуск бота и парсера вместе
async def main():
    # Запускаем и бота, и парсер одновременно
    await asyncio.gather(
        dp.start_polling(bot),
        sheet_watcher()
    )

if __name__ == '__main__':
    asyncio.run(main())
